# SuperKart Sales Forecasting and Model Deployment

**End-to-end regression project:** business understanding, data quality review, EDA, feature engineering, preprocessing pipelines, model comparison, hyperparameter tuning, serialization, and deployment-ready Flask and Streamlit applications.

> **Reproducibility note:** Keep `SuperKart.csv` in the same folder as this notebook (or upload it when prompted in Google Colab), then run all cells from top to bottom.

## 1. Business Context and Objective

SuperKart operates supermarkets and food marts across multiple city tiers. Accurate product–store sales forecasts can improve replenishment, inventory allocation, store planning, and regional sales decisions.

**Objective:** predict `Product_Store_Sales_Total` for a product–store combination and package the selected model as a reusable service.

**Decision framing:** This is a supervised regression problem. The model supports planning; it does not prove causal relationships. Because the supplied data has no transaction date, this notebook predicts held-out product–store records rather than performing a true chronological time-series forecast.

## 2. Setup and Imports

In [ ]:
# Uncomment only if your environment does not already contain these packages.
# %pip install -q numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 flask==3.1.1 streamlit==1.45.1 requests==2.32.4 huggingface_hub==0.34.0

import warnings, os, json, sys
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='viridis')
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)

## 3. Load the Dataset

In [ ]:
DATA_PATH = Path('SuperKart.csv')
if not DATA_PATH.exists():
    candidates = list(Path('.').rglob('SuperKart.csv'))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        try:
            from google.colab import files
            files.upload()
            DATA_PATH = Path('SuperKart.csv')
        except Exception as exc:
            raise FileNotFoundError('Place SuperKart.csv beside the notebook and rerun.') from exc

df = pd.read_csv(DATA_PATH)
print(f'Dataset path: {DATA_PATH.resolve()}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())

## 4. Data Overview and Quality Assessment

In [ ]:
overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'missing': df.isna().sum(),
    'missing_%': (100 * df.isna().mean()).round(2),
    'unique': df.nunique()
})
display(overview)
print('Duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)

### Data-overview observations

- The dataset contains **8,763 observations and 12 variables**; the target is continuous, confirming a regression task.
- There are **no missing values and no exact duplicate rows**, so no record deletion or statistical imputation is required for this version of the data. Imputers remain inside the deployment pipeline to make inference more resilient.
- `Product_Id` is unique for every row. Using the full identifier as a one-hot feature would create thousands of sparse columns and invite memorization. A lower-cardinality product-family prefix is engineered instead.
- Store identifiers and store establishment years each have only four levels. Their strong correspondence should be considered when interpreting importance; prediction importance is not causal importance.

## 5. Exploratory Data Analysis

In [ ]:
numeric_cols = ['Product_Weight','Product_Allocated_Area','Product_MRP','Store_Establishment_Year','Product_Store_Sales_Total']
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color=sns.color_palette('viridis', 6)[2])
    ax.set_title(f'Distribution of {col}')
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, ['Product_Weight','Product_Allocated_Area','Product_MRP']):
    sns.boxplot(x=df[col], ax=ax, color=sns.color_palette('magma', 5)[2])
    ax.set_title(f'Boxplot: {col}')
plt.tight_layout(); plt.show()

In [ ]:
categorical_cols = ['Product_Sugar_Content','Product_Type','Store_Size','Store_Location_City_Type','Store_Type']
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
for ax, col in zip(axes.flat, categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, palette='crest')
    ax.set_title(f'Count of records by {col}')
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))
sns.scatterplot(data=df, x='Product_MRP', y='Product_Store_Sales_Total', hue='Store_Type', alpha=.55, ax=axes[0,0])
axes[0,0].set_title('Sales vs MRP by Store Type')
sns.scatterplot(data=df, x='Product_Allocated_Area', y='Product_Store_Sales_Total', hue='Store_Size', alpha=.55, ax=axes[0,1])
axes[0,1].set_title('Sales vs Allocated Area by Store Size')
sns.boxplot(data=df, x='Store_Type', y='Product_Store_Sales_Total', palette='flare', ax=axes[1,0])
axes[1,0].tick_params(axis='x', rotation=20); axes[1,0].set_title('Sales Distribution by Store Type')
city_sales = df.groupby('Store_Location_City_Type', observed=True)['Product_Store_Sales_Total'].agg(['mean','median']).reset_index()
city_sales.plot(x='Store_Location_City_Type', y=['mean','median'], kind='bar', ax=axes[1,1], color=['#2A9D8F','#E76F51'])
axes[1,1].set_title('Mean and Median Sales by City Tier'); axes[1,1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(9,6)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0)
plt.title('Correlation Matrix'); plt.tight_layout(); plt.show()

display(df.groupby('Store_Type')['Product_Store_Sales_Total'].agg(['count','mean','median','std']).sort_values('mean', ascending=False).round(2))
display(df.groupby('Product_Type')['Product_Store_Sales_Total'].agg(['count','mean','median']).sort_values('mean', ascending=False).round(2))

In [ ]:
# Quantify outliers without automatically deleting valid high-value sales records.
outlier_summary=[]
for col in ['Product_Weight','Product_Allocated_Area','Product_MRP','Product_Store_Sales_Total']:
    q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1
    mask=(df[col] < q1-1.5*iqr) | (df[col] > q3+1.5*iqr)
    outlier_summary.append([col,int(mask.sum()),round(100*mask.mean(),2),round(q1-1.5*iqr,2),round(q3+1.5*iqr,2)])
display(pd.DataFrame(outlier_summary, columns=['variable','IQR_outliers','percent','lower_fence','upper_fence']))

### EDA insights and outlier decision

- Product MRP and allocated display area should be evaluated carefully because they are commercially plausible demand drivers.
- Sales vary across store formats and city tiers, supporting the use of both product and store attributes.
- IQR flags are **diagnostic, not automatic proof of bad data**. High sales can represent genuinely successful product–store combinations. Since the values are within plausible business ranges and tree ensembles are comparatively robust, records are retained. In production, validation rules should flag impossible values (negative weight/MRP/area or area above 1) rather than silently clipping legitimate demand.
- Category comparisons are descriptive and may be confounded by assortment, price, area, and store mix.

## 6. Data Preprocessing and Feature Engineering

**Engineering rationale:** `Product_Id` is converted to its two-letter family prefix. `Store_Age` is derived using the dataset's latest establishment year as a stable reference. The raw unique product identifier and raw establishment year are then removed. This avoids high-cardinality one-hot encoding and gives the model interpretable, reusable features.

The split is performed **before fitting preprocessing**, and all transformations live inside a pipeline to prevent leakage.

In [ ]:
TARGET='Product_Store_Sales_Total'

def engineer_features(frame):
    out=frame.copy()
    out['Product_Category_Code']=out['Product_Id'].astype(str).str[:2]
    reference_year=int(df['Store_Establishment_Year'].max())
    out['Store_Age']=reference_year-out['Store_Establishment_Year']
    return out.drop(columns=['Product_Id','Store_Establishment_Year'])

X_raw=df.drop(columns=TARGET)
y=df[TARGET]
X=engineer_features(X_raw)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=SEED)

num_features=X.select_dtypes(include=np.number).columns.tolist()
cat_features=X.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor=ColumnTransformer([('num',numeric_pipe,num_features),('cat',categorical_pipe,cat_features)], remainder='drop')

print('Train:',X_train.shape,'Test:',X_test.shape)
print('Numeric:',num_features)
print('Categorical:',cat_features)

## 7. Metric Selection and Baseline Models

**Primary metric: RMSE.** It is in the same units as sales and penalizes large forecasting misses more heavily—important when a large under-forecast can cause stock-outs or a large over-forecast can create excess inventory. **MAE** provides an easy-to-explain typical absolute error, while **R²** shows explained variance. MAPE is reported only as a secondary metric because it can become unstable when actual sales are close to zero.

In [ ]:
def regression_metrics(y_true,y_pred):
    return {'RMSE':mean_squared_error(y_true,y_pred)**0.5,
            'MAE':mean_absolute_error(y_true,y_pred),
            'R2':r2_score(y_true,y_pred),
            'MAPE_%':mean_absolute_percentage_error(y_true,y_pred)*100}

models={
 'Random Forest':RandomForestRegressor(n_estimators=250,min_samples_leaf=2,random_state=SEED,n_jobs=1),
 'Gradient Boosting':GradientBoostingRegressor(random_state=SEED)
}
fitted={}; rows=[]
for name,estimator in models.items():
    pipe=Pipeline([('preprocess',preprocessor),('model',estimator)])
    pipe.fit(X_train,y_train); fitted[name]=pipe
    rows.append({'Model':name,'Split':'Train',**regression_metrics(y_train,pipe.predict(X_train))})
    rows.append({'Model':name,'Split':'Test',**regression_metrics(y_test,pipe.predict(X_test))})
baseline_results=pd.DataFrame(rows)
display(baseline_results.round(3))
sns.barplot(data=baseline_results[baseline_results.Split=='Test'],x='Model',y='RMSE',palette='viridis')
plt.title('Baseline Test RMSE (lower is better)'); plt.show()

### Baseline interpretation

Compare train and test error together. A very low training error paired with materially higher test error indicates variance/overfitting. Gradient boosting is often more controlled, while random forest benefits from tuning tree depth, feature sampling, and minimum leaf size. Selection is based on held-out RMSE, not training fit.

## 8. Hyperparameter Tuning

`GridSearchCV` optimizes negative RMSE using three-fold cross-validation on the training set only. The final test set remains untouched until model comparison.

In [ ]:
searches={
 'Tuned Random Forest':(
  RandomForestRegressor(random_state=SEED,n_jobs=1),
  {'model__n_estimators':[180], 'model__max_depth':[None,12], 'model__min_samples_leaf':[1,3], 'model__max_features':[.8]}),
 'Tuned Gradient Boosting':(
  GradientBoostingRegressor(random_state=SEED),
  {'model__n_estimators':[120,180], 'model__learning_rate':[.03,.07], 'model__max_depth':[3], 'model__subsample':[.8]})
}
tuned={}; tuning_rows=[]
for name,(estimator,grid) in searches.items():
    pipe=Pipeline([('preprocess',preprocessor),('model',estimator)])
    gs=GridSearchCV(pipe,grid,scoring='neg_root_mean_squared_error',cv=3,n_jobs=1,return_train_score=True)
    gs.fit(X_train,y_train); tuned[name]=gs.best_estimator_
    tuning_rows.append({'Model':name,'Best CV RMSE':-gs.best_score_,'Best Parameters':gs.best_params_})
tuning_results=pd.DataFrame(tuning_rows)
display(tuning_results)
for _,r in tuning_results.iterrows(): print(r['Model'], '\n ', r['Best Parameters'])

## 9. Final Comparison, Test Evaluation, and Serialization

In [ ]:
comparison=[]
all_models={**fitted,**tuned}
for name,model in all_models.items():
    comparison.append({'Model':name,**regression_metrics(y_test,model.predict(X_test))})
comparison=pd.DataFrame(comparison).sort_values('RMSE').reset_index(drop=True)
display(comparison.round(3))

best_name=comparison.loc[0,'Model']; best_model=all_models[best_name]
test_pred=best_model.predict(X_test)
print('Selected model:',best_name)

fig,axes=plt.subplots(1,2,figsize=(15,5))
sns.scatterplot(x=y_test,y=test_pred,alpha=.6,ax=axes[0],color='#2A9D8F')
lims=[min(y_test.min(),test_pred.min()),max(y_test.max(),test_pred.max())]
axes[0].plot(lims,lims,'--',color='#E76F51'); axes[0].set(xlabel='Actual Sales',ylabel='Predicted Sales',title='Actual vs Predicted')
residuals=y_test-test_pred
sns.scatterplot(x=test_pred,y=residuals,alpha=.6,ax=axes[1],color='#6A4C93')
axes[1].axhline(0,ls='--',color='#E76F51'); axes[1].set(xlabel='Predicted Sales',ylabel='Residual',title='Residual Diagnostic')
plt.tight_layout(); plt.show()

MODEL_PATH='superkart_sales_pipeline.joblib'
joblib.dump(best_model,MODEL_PATH)
loaded_model=joblib.load(MODEL_PATH)
loaded_pred=loaded_model.predict(X_test)
assert np.allclose(test_pred,loaded_pred)
print(f'Serialized to {MODEL_PATH}; reload verification passed.')

In [ ]:
# Model-agnostic importance on the held-out set (measures predictive dependence, not causality).
perm=permutation_importance(best_model,X_test,y_test,n_repeats=8,random_state=SEED,scoring='neg_root_mean_squared_error',n_jobs=1)
importance=pd.DataFrame({'Feature':X_test.columns,'Importance':perm.importances_mean,'Std':perm.importances_std}).sort_values('Importance',ascending=False)
display(importance)
plt.figure(figsize=(10,6)); sns.barplot(data=importance.head(12),x='Importance',y='Feature',palette='rocket')
plt.title('Permutation Importance on Test Data'); plt.tight_layout(); plt.show()

### Selection conclusion

The model at the top of the comparison table is selected because it achieves the lowest untouched-test RMSE. The serialized object contains preprocessing and estimation together, preventing training-serving skew. The actual-vs-predicted and residual plots should be reviewed for systematic underprediction at high sales levels or changing error variance.

## 10. Deployment – Flask Backend

In [ ]:
from pathlib import Path
Path('backend_files').mkdir(exist_ok=True)
backend_app = r"""from flask import Flask, request, jsonify
import joblib, pandas as pd

app=Flask(__name__)
model=joblib.load('superkart_sales_pipeline.joblib')

def engineer(payload):
    frame=pd.DataFrame(payload if isinstance(payload,list) else [payload])
    frame['Product_Category_Code']=frame['Product_Id'].astype(str).str[:2]
    frame['Store_Age']=2009-frame['Store_Establishment_Year']
    return frame.drop(columns=['Product_Id','Store_Establishment_Year'])

@app.get('/health')
def health(): return jsonify({'status':'healthy'})

@app.post('/predict')
def predict():
    try:
        data=request.get_json(force=True)
        predictions=model.predict(engineer(data)).tolist()
        return jsonify({'predictions':[round(float(x),2) for x in predictions]})
    except Exception as exc:
        return jsonify({'error':str(exc)}),400

if __name__=='__main__': app.run(host='0.0.0.0',port=7860)
"""
Path('backend_files/app.py').write_text(backend_app)
Path('backend_files/requirements.txt').write_text('flask==3.1.1\npandas==2.2.2\nscikit-learn==1.6.1\njoblib==1.4.2\ngunicorn==23.0.0\n')
Path('backend_files/Dockerfile').write_text("""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 7860
CMD ["gunicorn", "--bind", "0.0.0.0:7860", "app:app"]
""")
Path('backend_files/superkart_sales_pipeline.joblib').write_bytes(Path(MODEL_PATH).read_bytes())
print('Backend files:',[p.name for p in Path('backend_files').iterdir()])

In [ ]:
# Local smoke test of the generated API (does not start a long-running server).
import importlib.util
spec=importlib.util.spec_from_file_location('backend_app',Path('backend_files/app.py'))
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
sample=df.drop(columns=TARGET).iloc[0].to_dict()
with module.app.test_client() as client:
    print('Health:',client.get('/health').get_json())
    print('Prediction:',client.post('/predict',json=sample).get_json())

## 11. Deployment – Streamlit Frontend

In [ ]:
Path('frontend_files').mkdir(exist_ok=True)
frontend_app=r"""import streamlit as st
import requests

st.set_page_config(page_title='SuperKart Sales Forecast',page_icon='🛒',layout='centered')
st.title('🛒 SuperKart Sales Forecast')
st.caption('Enter product and store details to estimate sales revenue.')
API_URL=st.secrets.get('API_URL','http://localhost:7860/predict')

with st.form('forecast'):
    product_id=st.text_input('Product ID','FD6114')
    weight=st.number_input('Product Weight',min_value=0.01,value=12.66)
    sugar=st.selectbox('Sugar Content',['Low Sugar','Regular','No Sugar'])
    area=st.slider('Allocated Area Ratio',0.0,1.0,0.027,0.001)
    ptype=st.selectbox('Product Type',['Frozen Foods','Dairy','Canned','Baking Goods','Health and Hygiene','Fruits and Vegetables','Snack Foods','Household','Soft Drinks','Meat','Hard Drinks','Breads','Breakfast','Seafood','Starchy Foods','Others'])
    mrp=st.number_input('Product MRP',min_value=0.01,value=117.08)
    store_id=st.selectbox('Store ID',['OUT001','OUT002','OUT003','OUT004'])
    year=st.selectbox('Establishment Year',[1987,1998,1999,2009])
    size=st.selectbox('Store Size',['Small','Medium','High'])
    city=st.selectbox('City Tier',['Tier 1','Tier 2','Tier 3'])
    store_type=st.selectbox('Store Type',['Departmental Store','Supermarket Type1','Supermarket Type2','Food Mart'])
    submitted=st.form_submit_button('Predict Sales',use_container_width=True)

if submitted:
    payload={'Product_Id':product_id,'Product_Weight':weight,'Product_Sugar_Content':sugar,'Product_Allocated_Area':area,'Product_Type':ptype,'Product_MRP':mrp,'Store_Id':store_id,'Store_Establishment_Year':year,'Store_Size':size,'Store_Location_City_Type':city,'Store_Type':store_type}
    try:
        response=requests.post(API_URL,json=payload,timeout=30); response.raise_for_status()
        st.success(f"Estimated sales: {response.json()['predictions'][0]:,.2f}")
    except Exception as exc: st.error(f'Prediction service error: {exc}')
"""
Path('frontend_files/app.py').write_text(frontend_app)
Path('frontend_files/requirements.txt').write_text('streamlit==1.45.1\nrequests==2.32.4\n')
Path('frontend_files/Dockerfile').write_text("""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]
""")
print('Frontend files:',[p.name for p in Path('frontend_files').iterdir()])

## 12. Hugging Face Spaces Upload (Optional, Credential-Gated)

Create two Spaces manually first: a **Docker** Space for the backend and a **Docker/Streamlit** Space for the frontend. Set the frontend secret `API_URL` to `https://<backend-space>.hf.space/predict`. Never hard-code an access token in the notebook.

The cell below runs only after you deliberately set the repository IDs and provide `HF_TOKEN` as an environment secret.

In [ ]:
# Optional deployment; leave RUN_HF_UPLOAD=False until both Space repositories exist.
RUN_HF_UPLOAD=False
BACKEND_REPO_ID='YOUR_USERNAME/superkart-backend'
FRONTEND_REPO_ID='YOUR_USERNAME/superkart-frontend'

if RUN_HF_UPLOAD:
    from huggingface_hub import HfApi
    token=os.environ.get('HF_TOKEN')
    if not token: raise ValueError('Store HF_TOKEN as an environment secret first.')
    api=HfApi(token=token)
    api.upload_folder(folder_path='backend_files',repo_id=BACKEND_REPO_ID,repo_type='space')
    api.upload_folder(folder_path='frontend_files',repo_id=FRONTEND_REPO_ID,repo_type='space')
    print('Backend:',f'https://huggingface.co/spaces/{BACKEND_REPO_ID}')
    print('Frontend:',f'https://huggingface.co/spaces/{FRONTEND_REPO_ID}')
else:
    print('Upload skipped safely. Set repository IDs, HF_TOKEN, and RUN_HF_UPLOAD=True when ready.')

## 13. Actionable Insights and Business Recommendations

1. **Use forecasts for replenishment prioritization.** Rank product–store combinations by predicted sales and combine the ranking with lead time, safety stock, shelf life, and margin. A revenue forecast alone should not directly determine order quantity.
2. **Treat MRP and allocated area as scenario levers, not causal guarantees.** Use the model to screen scenarios, then validate pricing and shelf-space changes with controlled store pilots because historical association may reflect assortment and store selection.
3. **Build store-format and city-tier planning views.** Aggregate predictions by store type and tier to support regional inventory and staffing discussions, while retaining product-level forecasts for replenishment.
4. **Investigate large residuals.** Product–store records with the largest absolute errors may indicate promotions, local events, stock-outs, data defects, or omitted seasonal effects. These are valuable candidates for root-cause analysis.
5. **Add time and operational signals.** The next model version should include transaction date, holidays, promotions, discounts, on-hand inventory, stock-out flags, competitor price, weather, and local events. Without dates, the current solution is not a true upcoming-quarter time-series forecast.
6. **Monitor production drift.** Track RMSE/MAE when actuals arrive, error by store/tier/product type, feature distribution drift, unknown categories, API latency, and failure rate. Define retraining triggers instead of relying only on a fixed schedule.
7. **Use decision-aware validation.** If future data includes dates, replace the random split with rolling/forward validation. Also consider grouped validation by store or product when the intended use includes unseen entities.
8. **Apply governance controls.** Log model version, request timestamp, input schema, prediction, and later actual outcome; validate ranges; protect the API with authentication and rate limiting; and document rollback procedures.

## 14. Final Conclusion, Limitations, and Reviewer Notes

This notebook delivers a reproducible end-to-end regression workflow. It preserves preprocessing inside the model pipeline, compares two ensemble families, tunes both using cross-validated RMSE, evaluates once on an untouched test set, verifies serialization, and generates deployable backend/frontend artifacts.

**Key limitation:** the business wording refers to next-quarter forecasting, but the supplied table has no time index. Therefore, results demonstrate cross-sectional sales prediction, not temporal extrapolation. This distinction should be stated clearly to reviewers—it prevents overstating the solution.

**Production-readiness gap:** a successful notebook and API smoke test do not by themselves prove production readiness. Before launch, add automated schema tests, container vulnerability scanning, authentication, centralized logging, monitoring, load tests, model/version registry, approval gates, and rollback capability.

**Suggested reviewer answer:** The final model is selected by the lowest untouched-test RMSE because large revenue misses are operationally costly. MAE and R² are retained as complementary measures, and subgroup/drift monitoring is recommended because a single global score can hide weak performance in particular store or product segments.